In [1]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter

/media/ashish/10EE7C4EEE7C2E5A/youtube_chatbot/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## transcript loading from YouTubeTranscriptApi


In [2]:
def format_transcript(fetched_transcript):
    return "".join(snippet.text for snippet in fetched_transcript)

In [22]:
import re
def extract_video_id(url):
    pattern = r'(?:youtube\.com\/(?:.*v=|.*\/|embed\/|shorts\/)|youtu\.be\/)([a-zA-Z0-9_-]{11})'
    match = re.search(pattern, url)
    
    if match:
        return match.group(1)
    return None


In [25]:
url = "https://www.youtube.com/watch?v=TYLZG9tv0Yw"
video_id = extract_video_id(url=url)
video_id

'TYLZG9tv0Yw'

In [26]:
ytt_api = YouTubeTranscriptApi()
try:
    fetched_transcript = ytt_api.fetch(video_id=video_id)
except :
    print("coulnot find transcript")

In [27]:

transcript = format_transcript(fetched_transcript=fetched_transcript)
transcript

"[Music]50 million years ago, two continentscollided with unstoppable force.India slammed into Asia, changing theplanet forever.[Music]From that impact rose the Himalayas,the world's highest peaks.[Music]In this thin air, Nepal was born. Anation unlike any other.Prehistoric giants still roam itsgrasslands.Isolated villages cling to its slopes.And above it all, the highest point onEarth touches the edge of space.Eight giants rise from Nepal's borders.[Music]The world's 10 tallest peaks and eightcall this place home.[Music]A wall of stone and ice that interceptsthe monsoons,births three of Asia's great rivers,and shapes weather patterns across halfthe planet.Everest base camp sits at 17,600ft, higher than most mountains will everreach.[Music]Yet this is only the beginning.[Music]Colorful tents scatter across the glacialike prayer flags. A temporary citywhere climbers spend weeks preparingbodies and minds for what lies above.[Music]The ascent begins in landscapes thatfeel borrowed from an

# text splitting

In [5]:

text_splitter= RecursiveCharacterTextSplitter(chunk_size= 200 , chunk_overlap = 50 )
text_splitter

In [6]:
documents= text_splitter.create_documents([transcript])
len(documents)

166

In [7]:
documents[0]

Document(metadata={}, page_content="[Music]50 million years ago, two continentscollided with unstoppable force.India slammed into Asia, changing theplanet forever.[Music]From that impact rose the Himalayas,the world's highest")

# Embedding the chunks and storing 

In [8]:
from langchain_huggingface import  HuggingFaceEndpointEmbeddings
from langchain_community.vectorstores import Chroma

In [9]:
from dotenv import  load_dotenv
import os
load_dotenv()

huggingface_api_token = os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")

In [10]:
embeddings = HuggingFaceEndpointEmbeddings(repo_id='BAAI/bge-large-en-v1.5',huggingfacehub_api_token=huggingface_api_token)

In [11]:
vector_store = Chroma(embedding_function=embeddings,collection_name="collection",persist_directory='./my_vectors')

/tmp/ipykernel_27388/1482599673.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(embedding_function=embeddings,collection_name="collection",persist_directory='./my_vectors')


In [12]:
vector_store.add_documents(documents=documents)

['1632c3d2-c40c-4ee6-badb-610a2d3a1929',
 '25cf1059-94cf-4346-9593-12e3fecf82c9',
 '050360a4-72bf-43aa-8485-8ef8b39b80c4',
 '059b11ee-5080-4bf5-aaa6-d43751f2785a',
 '31f4dac9-2cb2-41fd-b4ad-199e3dc9d796',
 '876cf435-55fe-49fb-bbcc-aed9578f2fd0',
 'dd6cb423-0eda-43e3-a102-301261ee4990',
 'accf8e31-faf8-4c87-82f6-57cc96a2687d',
 '05cbfd7c-6592-4783-a64d-b6e253a4fb6c',
 'bd7c6f9a-f4ec-47c7-9fb1-64e2c409e163',
 '9a1c9ff4-022e-4c3c-ab2d-b5b55c06b9d8',
 '4a68a907-41c8-4d4a-89d8-a32d202a7365',
 '801405ca-49d1-44b7-b570-2ab92325b755',
 'a44ef026-2760-4540-b7a6-5caa7e93940b',
 'ea29cd0c-8fb4-4191-a41f-f310986bf00d',
 '88f40b3f-945c-4112-94ef-1e4e2eba6710',
 '1b740cec-0c63-482d-8e0a-adf1be4cb9dd',
 'eaf300bb-20b6-47d2-971b-df980e4676a2',
 '4f33162e-64f3-416c-98cc-389429ccc26f',
 'ba514888-6cd4-4cbd-b0f5-5f923e5718f1',
 'c2d0b531-4453-4f57-96ac-9991c459d885',
 'a439e901-e95a-4862-869f-e9c63a5e478f',
 'f2f61382-f6ff-441f-baa4-4c3520991b1b',
 '2c50279a-41a9-4dea-b56e-ba86128c770b',
 '019ec62b-8096-

In [13]:
# vector_store.get(['03b67e76-11ff-4d26-970f-5776a9f930d8'])

In [14]:
query="what are the evidence that the nepal is the oldest country?"
retriver =vector_store.as_retriever(search_type='mmr',search_kwargs={'k':4})

In [15]:
rel_docs= retriver.invoke(query)

# prompt

In [16]:
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import ChatHuggingFace , HuggingFaceEndpoint

In [17]:
prompt_template = PromptTemplate(
    template='''
    You are a helpful assistant.
    Answer only from the provided transcript context.
    If the context is insufficient just say you don't know.
    {context}
    question:{question}
       '''
)

In [18]:
llm = HuggingFaceEndpoint(repo_id='meta-llama/Llama-3.1-8B-Instruct',huggingfacehub_api_token=huggingface_api_token)

In [19]:
model = ChatHuggingFace(llm=llm)

In [20]:
prompt = prompt_template.invoke({'context':rel_docs,'question':query})

In [21]:
model.invoke(prompt).content

'According to the provided transcript, the evidence that Nepal is the oldest country is mentioned as: "Nepal remains the oldest country in South Asia, never having fallen"'